# Projet 9 — Petit LLM : fine-tuning QLoRA pour l'extraction d'informations en JSON (variante A)

**Question** : un petit modèle ouvert (Qwen2.5-1.5B-Instruct, 4 bits) adapté par QLoRA extrait-il mieux
les entités (personnes, organisations, lieux) au format JSON que **le même modèle avec prompt seul**
(zero-shot et few-shot), sur un jeu de test jamais vu ?

* Toute la logique est dans le paquet `src/llmx` (testé) ; ce notebook ne fait qu'orchestrer.
* Tous les hyperparamètres sont dans `configs/base.yaml`.
* Les choix (nombre d'époques, meilleure baseline) se font **sur la validation** par une règle fixée
  à l'avance (section 4) ; le **test n'est utilisé qu'une fois** par système (section 6, verrou logiciel).
* Exécution de développement archivée : `notebooks/archive/2026-11-27_validation_run.ipynb`.

## 0. Installation et environnement

In [ ]:
import os, sys, pathlib

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # un seul GPU (Kaggle fournit parfois 2 x T4)
REPO_URL = "https://github.com/Aziz-Gnouma/llm-extract.git"
# Kaggle se reconnaît à sa variable d'environnement (Colab possède aussi un dossier /kaggle)
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
BASE = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
ROOT = BASE / "llm-extract"
if not ROOT.exists():
    !git clone -q {REPO_URL} {ROOT}
%cd {ROOT}
!pip install -q -e ".[ml,dev]"
sys.path.insert(0, str(ROOT / "src"))   # évite un redémarrage du noyau après l'installation editable

In [ ]:
import json, platform, subprocess, importlib.metadata as md
import torch

env = {"python": platform.python_version(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "git_commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
       **{p: md.version(p) for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "datasets"]}}
print(json.dumps(env, indent=2))
assert env["gpu"], "Activez un GPU (T4) dans les paramètres du notebook"
pathlib.Path("reports").mkdir(exist_ok=True)
pathlib.Path("reports/environment.json").write_text(json.dumps(env, indent=2))

In [ ]:
!python -m pytest -q   # tests unitaires : parsing, métriques, découpages, masquage des labels

## 1. Données : CoNLL-2003 → JSON

Chaque phrase devient `{"persons": [...], "organizations": [...], "locations": [...]}` (MISC est écarté).
Découpage **figé** : test tiré du *test* officiel, validation du *validation*, entraînement du *train* ;
doublons supprimés et aucune phrase partagée entre ensembles. Les fichiers régénérés doivent reproduire
exactement les empreintes SHA-256 de `data/processed/manifest.json` (sinon erreur).

In [ ]:
import pandas as pd
from llmx.config import load_config, set_seed
from llmx import pipeline

cfg = load_config("configs/base.yaml")
set_seed(cfg.seed)
manifest = pipeline.prepare_data(cfg)
splits = pipeline.load_splits(cfg)
pd.DataFrame({k: {"n": v["n"], **v["entity_counts"], "sha256": v["sha256"][:12]}
              for k, v in manifest["splits"].items()}).T

In [ ]:
import matplotlib.pyplot as plt

lengths = {k: [len(r["text"].split()) for r in v] for k, v in splits.items()}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
counts = pd.DataFrame({k: v["entity_counts"] for k, v in manifest["splits"].items()}).T
(counts.drop(columns="sentences_without_entity").div(counts.sum(axis=1), axis=0)
       .plot.bar(ax=ax[0], title="Répartition des entités par ensemble", rot=0))
ax[1].hist([lengths["train"], lengths["test"]], bins=30, label=["train", "test"], density=True)
ax[1].set_title("Longueur des phrases (tokens)"); ax[1].legend()
plt.tight_layout(); plt.savefig("reports/data_overview.png", dpi=120)
for r in splits["train"][:3]: print(r["text"], "\n  →", r["entities"])

## 2. Baselines : même modèle, prompt seul (validation)

Le prompt système est **identique** pour toutes les variantes ; le few-shot ajoute k exemples tirés
du *train* uniquement. Baseline triviale à garder en tête : répondre toujours « vide » donne
une exact match égale à la part de phrases sans entité (~21 %) et un F1 de 0.

In [ ]:
zero = pipeline.run_evaluation(cfg, "zero_shot", "val")
k = cfg.baseline.few_shot_k
few = pipeline.run_evaluation(cfg, f"few_shot{k}", "val", k_shots=k)
pd.DataFrame(pipeline.collect_metrics(cfg)).round(3)

## 3. Fine-tuning QLoRA (configuration principale : 1 500 exemples, 2 époques)

In [ ]:
report = pipeline.run_training(cfg)
print({k: v for k, v in report.items() if k != "log_history"})

In [ ]:
def plot_curves(report, title, path):
    log = pd.DataFrame(report["log_history"])
    fig, ax = plt.subplots(figsize=(7, 3.5))
    log.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train loss")
    log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o", label="val loss")
    ax.set_title(title); plt.tight_layout(); plt.savefig(path, dpi=120); plt.show()

plot_curves(report, "Courbes d'apprentissage (2 époques)", f"reports/{cfg.run_name}/learning_curves.png")

In [ ]:
qlora = pipeline.run_evaluation(cfg, "qlora", "val", use_adapter=True)
ORDER = ["zero_shot", f"few_shot{k}", "qlora"]
table = pd.DataFrame(pipeline.collect_metrics(cfg)).query("split == 'val'").set_index("system").loc[ORDER]
table.round(3)

In [ ]:
def plot_f1(table, title, path):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    f1 = table["micro_f1"].astype(float)
    lo = f1 - table["f1_ci95"].map(lambda c: float(c[0]))
    hi = table["f1_ci95"].map(lambda c: float(c[1])) - f1
    ax.bar(table.index, f1, yerr=[lo, hi], capsize=6, color=["#9aa5b1", "#7b8794", "#2563eb"])
    for i, (v, top) in enumerate(zip(f1, f1 + hi)):
        ax.text(i, top + 0.02, f"{v:.2f}", ha="center", va="bottom", fontweight="bold")
    ax.set_ylabel("micro-F1 (IC bootstrap 95 %)"); ax.set_ylim(0, 1.05); ax.set_title(title)
    plt.tight_layout(); plt.savefig(path, dpi=120); plt.show()

plot_f1(table, "Validation", f"reports/{cfg.run_name}/f1_val.png")

## 4. Expériences de réglage et choix de la configuration finale (validation uniquement)

Chaque variante est une copie de la configuration avec un autre `run_name` : ses artefacts sont séparés.

* **4.a Quantité de données** : 100 / 500 / 1 500 exemples (`train.max_train_examples`).
* **4.b Durée d'entraînement** : 3 époques contre 2 (la perte de validation baissait encore à la fin
  de l'exécution de développement).
* **4.c Règle de choix, fixée avant de voir le test** : la configuration finale est celle des deux
  (2 ou 3 époques) qui a le **meilleur micro-F1 de validation** ; à égalité, la plus courte.

In [ ]:
from dataclasses import replace

def variant(name, **train_overrides):
    return replace(cfg, run_name=f"{cfg.run_name}-{name}", train=replace(cfg.train, **train_overrides))

ablation = []
for n in (100, 500):
    c = variant(f"n{n}", max_train_examples=n)
    pipeline.run_training(c)
    m = pipeline.run_evaluation(c, "qlora", "val", use_adapter=True)
    ablation.append({"n_train": n, "micro_f1": m["micro"]["f1"], "ci95": m["micro_f1_ci95"],
                     "hallucination": m["hallucination_rate"]})
ablation.append({"n_train": len(splits["train"]), "micro_f1": qlora["micro"]["f1"],
                 "ci95": qlora["micro_f1_ci95"], "hallucination": qlora["hallucination_rate"]})
ablation = pd.DataFrame(ablation)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(ablation["n_train"], ablation["micro_f1"], marker="o", capsize=5,
            yerr=[ablation["micro_f1"] - ablation["ci95"].map(lambda c: c[0]),
                  ablation["ci95"].map(lambda c: c[1]) - ablation["micro_f1"]], label="QLoRA")
ax.axhline(few["micro"]["f1"], ls="--", color="gray", label=f"few-shot ({k} ex.)")
ax.axhline(zero["micro"]["f1"], ls=":", color="gray", label="zero-shot")
ax.set_xscale("log"); ax.set_xticks(ablation["n_train"]); ax.set_xticklabels(ablation["n_train"])
ax.set_xlabel("exemples d'entraînement"); ax.set_ylabel("micro-F1 validation"); ax.set_ylim(0, 1)
ax.legend(); ax.set_title("Effet de la quantité de données")
plt.tight_layout(); plt.savefig(f"reports/{cfg.run_name}/data_size.png", dpi=120); plt.show()
ablation.round(3)

In [ ]:
cfg_e3 = variant("e3", epochs=3)
report_e3 = pipeline.run_training(cfg_e3)
qlora_e3 = pipeline.run_evaluation(cfg_e3, "qlora", "val", use_adapter=True)
plot_curves(report_e3, "Courbes d'apprentissage (3 époques)", f"reports/{cfg_e3.run_name}/learning_curves.png")

In [ ]:
candidates = {"2 époques": (cfg, qlora, report), "3 époques": (cfg_e3, qlora_e3, report_e3)}
summary = pd.DataFrame({name: {"micro_f1": m["micro"]["f1"], "ci95": tuple(round(x, 3) for x in m["micro_f1_ci95"]),
                               "best_val_loss": r["best_eval_loss"], "train_min": r["train_seconds"] / 60}
                        for name, (c, m, r) in candidates.items()}).T
display(summary)

# Règle 4.c : meilleur F1 de validation ; max() garde le premier (2 époques) en cas d'égalité
final_name = max(candidates, key=lambda name: candidates[name][1]["micro"]["f1"])
final_cfg, final_val, final_report = candidates[final_name]
print(f"Configuration finale retenue : {final_name}  (run '{final_cfg.run_name}')")

## 5. Analyse d'erreurs (validation, modèle final)

Taxonomie : `invalid_json`, `hallucination` (valeur absente du texte), `wrong_field` (bonne entité,
mauvais type), `boundary` (frontière trop longue/courte), `spurious` (pas une entité), `missed`.

In [ ]:
from collections import Counter
from llmx.data.io import read_json, read_jsonl

def val_dir(c, system): return c.run_reports_dir / f"{system}_val"
systems = {"zero_shot": cfg, f"few_shot{k}": cfg, "qlora": final_cfg}
errors = pd.DataFrame({s: read_json(val_dir(c, s) / "metrics.json")["error_summary"] for s, c in systems.items()}).T
fields = pd.DataFrame({s: {f: v["f1"] for f, v in read_json(val_dir(c, s) / "metrics.json")["per_field"].items()}
                       for s, c in systems.items()}).T
display(errors); display(fields.round(3))

In [ ]:
preds = read_jsonl(val_dir(final_cfg, "qlora") / "predictions.jsonl")

# Confusions de type : champ de référence -> champ prédit
confusions = Counter()
for p in preds:
    for e in p["errors"]:
        if e["category"] == "wrong_field":
            gold = next((f for f, vs in p["gold"].items() if e["value"] in [v.lower() for v in vs]), "?")
            confusions[f"{gold} -> {e['field']}"] += 1
print("Confusions de type :", dict(confusions))

# Titres en majuscules (dépêches sportives) contre le reste
def mostly_upper(t):
    words = [w for w in t.split() if w.isalpha()]
    return bool(words) and sum(w.isupper() for w in words) > 0.6 * len(words)
upper = [p for p in preds if mostly_upper(p["text"])]
rate = lambda rows: sum(bool(p["errors"]) for p in rows) / max(1, len(rows))
print(f"Phrases en majuscules : {len(upper)}, taux d'erreur {rate(upper):.0%} contre {rate(preds):.0%} au total")

In [ ]:
wrong = [p for p in preds if p["errors"]]
print(f"{len(wrong)} phrases avec au moins une erreur sur {len(preds)}\n")
for p in wrong[:20]:
    print("TEXTE :", p["text"])
    print("  gold :", p["gold"])
    print("  pred :", p["pred"])
    print("  ->", [(e["category"], e["field"], e["value"]) for e in p["errors"]], "\n")

## 6. Évaluation finale sur le test — une seule fois par système

La configuration finale a été choisie en section 4 sur la validation. Les trois systèmes sont évalués
une fois chacun sur les 500 phrases de test ; une seconde exécution lève `HoldoutAlreadyUsedError`
(verrous dans `reports/test_locks/`).

In [ ]:
RUN_TEST = True   # exécution de remise finale

if RUN_TEST:
    test = {
        "zero_shot": pipeline.run_evaluation(cfg, "zero_shot", "test"),
        f"few_shot{k}": pipeline.run_evaluation(cfg, f"few_shot{k}", "test", k_shots=k),
        "qlora": pipeline.run_evaluation(final_cfg, "qlora", "test", use_adapter=True),
    }
    test_table = pd.DataFrame({
        s: {"micro_f1": m["micro"]["f1"], "ci95": tuple(round(x, 3) for x in m["micro_f1_ci95"]),
            "precision": m["micro"]["precision"], "recall": m["micro"]["recall"],
            "exact_match": m["exact_match"], "strict_json": m["strict_json_rate"],
            "schema_valid": m["schema_valid_rate"], "hallucination": m["hallucination_rate"],
            "s_per_example": m["seconds_per_example"]}
        for s, m in test.items()}).T
    display(test_table)
    plot_f1(test_table.assign(f1_ci95=test_table["ci95"]), "Test (500 phrases, évalué une fois)",
            f"reports/{final_cfg.run_name}/f1_test.png")
else:
    print("Test non évalué (RUN_TEST = False) : résultats de développement sur la validation uniquement.")

## 7. Analyse critique

> Brouillon rédigé à partir de l'exécution de développement (validation, 27/11). Les valeurs **test**
> sont à reporter depuis la section 6 ; les valeurs de validation peuvent varier légèrement d'une
> exécution à l'autre (non-déterminisme du GPU).

### 7.1 Réponse à la question
Oui, et nettement. Sur la validation, le modèle adapté par QLoRA atteint un micro-F1 de **0,87**
(IC 95 % : 0,84–0,90) contre **0,61** (0,56–0,65) pour le few-shot et **0,38** (0,33–0,44) pour le
zero-shot. Les intervalles de confiance ne se chevauchent pas : l'écart n'est pas un effet de
l'échantillon. L'exact match (toutes les entités d'une phrase correctes) passe de 0,30 à 0,73.
Le gain s'obtient en entraînant **1,2 %** des paramètres (18,5 M sur 1,5 G) pendant environ
**13 minutes** sur un T4 gratuit ; l'adaptateur pèse 72 Mo.

### 7.2 Le format compte autant que l'exactitude
* **Zero-shot** : le modèle entoure systématiquement son JSON d'un bloc Markdown (```` ```json ````) :
  0 % de JSON strict. Sans analyseur tolérant, aucune réponse ne serait exploitable par un programme.
* **Few-shot** : JSON strict, mais **28 %** des réponses violent le schéma (clés vides omises, une clé
  `"events"` inventée) alors que les trois exemples montrent le schéma exact.
* **QLoRA** : 100 % de JSON strict et de schéma valide. Le fine-tuning apprend le *contrat de sortie*,
  pas seulement la tâche.

### 7.3 Hallucinations
Le taux de valeurs absentes du texte passe de 14,7 % (zero-shot) à 6,2 % (few-shot) puis 1,5 % (QLoRA).
* En zero-shot, ce sont surtout des **inférences** : « India » pour *Bombay Commodities*, « Libya »
  pour *Libyans*, « Israel » pour *Israeli crews*, un pays déduit d'un tableau de football. Le modèle
  raisonne avec ses connaissances au lieu d'extraire.
* Pour QLoRA, une partie des cas restants sont des **faux positifs de la métrique** : la comparaison
  se fait par mots entiers et CoNLL garde les mots composés en un seul jeton (« New York-based »,
  « Moscow-backed », « aid-U.N. »). Le vrai taux est donc encore plus bas. Restent de vraies
  reformulations (« Boris Nikolayevich Yeltsin » reconstruit à partir de « Boris Nikolayevich ( Yeltsin ) »).

### 7.4 Où le modèle se trompe encore
* Les oublis (`missed`) dominent : le modèle final est plus **précis** (0,91) que **complet** (0,84).
* Les **personnes** sont faciles (F1 0,95) ; **organisations** et **lieux** plafonnent à 0,81.
  Les confusions viennent surtout des conventions d'annotation : un pays ou une ville désigne une
  équipe (« ROMANIA BEAT LITHUANIA », « Bristol » en cricket), un joueur s'appelle « Washington ».
* Les **titres en majuscules** (dépêches sportives) sont deux fois plus souvent fautifs
  (10 phrases sur 17, contre 27 % au total) : la casse est un indice fort que le modèle perd.
* Une partie des faux positifs vient de **notre choix d'écarter MISC** : le modèle extrait
  « FED CUP », « GREATER MILWAUKEE OPEN » ou « Dow Jones », qui sont des MISC dans CoNLL.

### 7.5 Quantité de données et coût
Avec 100 exemples, le F1 (0,59) rejoint presque le few-shot (0,61, IC qui se recouvrent) ; 500 exemples donnent 0,73 et 1 500 donnent 0,87.
La courbe ne plafonne pas : davantage de données annotées serait le levier le plus rentable.
Coût à l'inférence : 0,37 s par phrase contre 0,19 s sans adaptateur, car l'adaptateur n'est pas
fusionné dans le modèle ; une fusion (`merge_and_unload`) supprimerait ce surcoût.

### 7.6 Limites
* **Un seul domaine** : dépêches Reuters en anglais de 1996. Rien ne garantit la généralisation à
  d'autres langues, époques ou registres (réseaux sociaux, documents administratifs).
* **Une seule graine** : les IC bootstrap mesurent la variabilité due à l'échantillon de phrases,
  pas celle de l'entraînement.
* **Validation optimiste** : elle a servi à choisir le point de contrôle et la configuration ;
  seul le test donne une estimation non biaisée.
* **Baselines non optimisées** : le prompt n'a pas été travaillé (pas de recherche de prompt,
  exemples few-shot tirés au hasard) ; l'écart mesuré est un majorant face à un prompt soigné.
* **Évaluation par chaîne exacte** (après minuscules et espaces) : une variante correcte
  (« U.S. » / « US ») compte comme une erreur ; étiquettes CoNLL elles-mêmes partiellement bruitées.
* **MISC écarté** : simplifie la tâche mais crée des faux positifs (7.4).

### 7.7 Éthique et risques
* **Données personnelles** : extraire automatiquement des noms de personnes relève du RGPD dès qu'on
  l'applique à des documents réels ; un tel outil peut servir au profilage ou à la surveillance.
* **Biais** : le corpus surreprésente l'actualité occidentale, politique et sportive de 1996, et des
  personnes majoritairement masculines ; les performances sur d'autres noms (non occidentaux,
  translittérés) ne sont pas mesurées.
* **Confiance trompeuse** : un JSON toujours valide inspire confiance alors que 16 % des entités
  sont encore manquées et que des hallucinations subsistent ; une validation humaine reste nécessaire.
* **Licences et sobriété** : modèle sous licence Apache 2.0 ; textes CoNLL/Reuters non redistribués
  dans le dépôt (seules les empreintes le sont) ; environ 40 minutes de GPU T4 pour l'ensemble des
  entraînements.

## 8. Sauvegarde des résultats

In [ ]:
# Archive des métriques, figures, journaux et adaptateurs LoRA (à recopier dans le dépôt)
!cd {ROOT} && zip -qr {BASE}/results.zip reports outputs/*/adapter && ls -lh {BASE}/results.zip
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(BASE / "results.zip"))
else:
    print("Kaggle : results.zip est disponible dans l'onglet Output du notebook.")